# CS781 Capstone, Phase 2: Legal Named Entity Recognition
**Darsh Kedia, 250311.** Codabench username `d_250311`.

Task: tag 12 entity types (APP, RESP, A.COUNSEL, R.COUNSEL, JUDGE, WIT, AUTH, COURT, STAT, PREC, DATE, CASENO) with character offsets in Indian court judgments. Metric: strict Macro-F1 over all 12 labels.

Pipeline: byte-level BPE tokenizer with character offsets, encoder, linear BIO head, sliding windows with averaged log-probs, BIO-constrained Viterbi decoding, spans, `prediction.json`, `submission.zip`.

Everything is trained from random initialisation on the official data only. No pretrained weights, no external corpora, no LLM APIs. Once a Phase 1 checkpoint exists, set `PHASE1_TOKENIZER` / `PHASE1_CKPT` below and use the hook in section 7.

Run on Kaggle with a GPU (T4 or P100). Needs the Kaggle secret `HF` holding a Hugging Face token that has accepted the dataset's terms.

In [ ]:
# ---- Config ----
SEED = 42
PHASE1_TOKENIZER = None   # e.g. "/kaggle/input/datasets/darshkedia/cs781-phase1/tokenizer.json"
PHASE1_CKPT = None        # e.g. "/kaggle/input/datasets/darshkedia/cs781-phase1/model_final.pt"
PHASE1_CODE_DIR = None    # folder holding cs781_p1.py from the Phase 1 notebook
ENCODER = "bilstm"        # "bilstm" (fold2 0.386 on CPU) or "transformer" (0.117: too little data from scratch)
DEV_EPOCHS = 30
RUN_NAME = f"{ENCODER}_scratch"
WORK = "/kaggle/working"

In [ ]:
!pip -q install tokenizers huggingface_hub

## 1. Helper modules
`lner_utils.py`: loading, span/BIO conversion, windows, strict scorer (matches the official `evaluate_phase2.py`), validator, zipper. `lner_model.py`: tokenizer, encoders, tagger, training and inference. `baseline_rules.py`: non-neural gazetteer + regex baseline.

In [ ]:
%%writefile lner_utils.py
"""Phase 2 (L-NER) utilities: data loading, char-offset <-> token BIO alignment,
strict span scoring, prediction.json validation and Codabench zip packaging.

Pure Python (no torch / transformers needed) so it works with any tokenizer:
a tokenizer only has to give (start_char, end_char) offsets for each token.

CLI:
  python lner_utils.py roundtrip TEXT.json GOLD.json   # ceiling F1 of word-level BIO alignment
  python lner_utils.py score GOLD.json PRED.json        # strict macro P/R/F1 (local held-out fold)
  python lner_utils.py validate TEST_TEXT.json PRED.json
  python lner_utils.py zip PRED.json [OUT.zip]          # prediction.json at zip root
"""

import json
import re
import sys
import zipfile
from collections import defaultdict

LABELS = ["APP", "RESP", "A.COUNSEL", "R.COUNSEL", "JUDGE", "WIT",
          "AUTH", "COURT", "STAT", "PREC", "DATE", "CASENO"]


# ----------------------------------------------------------------------------
# Loading
# ----------------------------------------------------------------------------

def load_texts(path):
    """{doc_id: text}. Also accepts a list of {"id", "text"} records."""
    with open(path, encoding="utf-8") as f:
        data = json.load(f)
    if isinstance(data, list):
        data = {str(d["id"]): d["text"] for d in data}
    return {str(k): v for k, v in data.items()}


def _norm_span(e):
    """Accept [s, e, label(, text)] or {"start","end","label"} style entries."""
    if isinstance(e, dict):
        s = e.get("start", e.get("start_char", e.get("begin")))
        t = e.get("end", e.get("end_char"))
        lab = e.get("label", e.get("type", e.get("entity")))
        return int(s), int(t), str(lab)
    return int(e[0]), int(e[1]), str(e[2])


def load_spans(path):
    """{doc_id: [(start, end, label), ...]} from a gold or prediction file."""
    with open(path, encoding="utf-8") as f:
        data = json.load(f)
    if isinstance(data, list):  # list of {"id", "entities"/"spans"/"pred"}
        data = {str(d["id"]): d.get("entities", d.get("spans", d.get("pred", []))) for d in data}
    return {str(k): [_norm_span(e) for e in v] for k, v in data.items()}


def load_labels(path):
    with open(path, encoding="utf-8") as f:
        return [l.strip() for l in f if l.strip()]


# ----------------------------------------------------------------------------
# Pre-tokenization with character offsets
# ----------------------------------------------------------------------------

# Words, numbers and single punctuation marks each become a token. Splitting
# punctuation off matters: gold spans like "Section 302" or "(2003) 5 SCC 123"
# often end right before a comma/full stop.
_WORD_RE = re.compile(r"\w+|[^\w\s]", re.UNICODE)


def word_offsets(text):
    """[(start, end), ...] for words/punctuation in text (whitespace dropped)."""
    return [(m.start(), m.end()) for m in _WORD_RE.finditer(text)]


def subword_offsets(text, encode_word):
    """Offsets for a subword tokenizer that works word by word.

    encode_word(word) -> list of token strings whose concatenated *character*
    lengths equal len(word) (true for most BPE/WordPiece once markers such as
    '##' or '</w>' are stripped). Returns [(tok, start, end), ...].
    """
    out = []
    for s, e in word_offsets(text):
        pos = s
        for tok in encode_word(text[s:e]):
            ln = len(tok)
            out.append((tok, pos, pos + ln))
            pos += ln
        if pos != e:
            raise ValueError(f"subword lengths do not cover word {text[s:e]!r}")
    return out


# ----------------------------------------------------------------------------
# Spans <-> BIO tags
# ----------------------------------------------------------------------------

def spans_to_bio(offsets, spans):
    """Tag each token B-X / I-X / O. A token belongs to a span if it overlaps it.

    Returns (tags, misaligned) where misaligned lists spans whose edges do not
    fall on token boundaries; those can never be predicted exactly from tags.
    """
    tags = ["O"] * len(offsets)
    misaligned = []
    starts = {s for s, _ in offsets}
    ends = {e for _, e in offsets}
    for s, e, lab in sorted(spans):
        if s not in starts or e not in ends:
            misaligned.append((s, e, lab))
        first = True
        for i, (ts, te) in enumerate(offsets):
            if te <= s or ts >= e:
                continue
            if tags[i] != "O":  # overlapping gold spans: keep the earlier one
                continue
            tags[i] = ("B-" if first else "I-") + lab
            first = False
    return tags, misaligned


def bio_to_spans(offsets, tags):
    """Decode BIO tags to [(start, end, label)]. A stray I-X starts a new span."""
    spans = []
    cur = None
    for (ts, te), tag in zip(offsets, tags):
        if tag == "O":
            if cur:
                spans.append(tuple(cur)); cur = None
            continue
        pre, lab = tag.split("-", 1)
        if pre == "B" or cur is None or cur[2] != lab:
            if cur:
                spans.append(tuple(cur))
            cur = [ts, te, lab]
        else:
            cur[1] = te
    if cur:
        spans.append(tuple(cur))
    return spans


def windows(n_tokens, max_len, stride):
    """Sliding windows [(lo, hi)] covering n_tokens; overlap = max_len - stride.
    When predicting, keep each token's label from the window where it is
    furthest from an edge (or average logits over windows)."""
    if n_tokens <= max_len:
        return [(0, n_tokens)]
    out, lo = [], 0
    while True:
        hi = min(lo + max_len, n_tokens)
        out.append((lo, hi))
        if hi == n_tokens:
            return out
        lo += stride


# ----------------------------------------------------------------------------
# Strict scoring
# ----------------------------------------------------------------------------

def strict_scores(gold, pred, labels=LABELS):
    """Strict match on (doc, start, end, label). Macro = mean over `labels` of
    per-label P/R/F1 over all 12 labels, matching the official
    evaluate_phase2.py: a label with no gold and no predictions scores 0."""
    tp, fp, fn = defaultdict(int), defaultdict(int), defaultdict(int)
    for doc in set(gold) | set(pred):
        g = set(gold.get(doc, []))
        p = set(pred.get(doc, []))
        for s in p & g: tp[s[2]] += 1
        for s in p - g: fp[s[2]] += 1
        for s in g - p: fn[s[2]] += 1
    per = {}
    for lab in labels:
        P = tp[lab] / (tp[lab] + fp[lab]) if tp[lab] + fp[lab] else 0.0
        R = tp[lab] / (tp[lab] + fn[lab]) if tp[lab] + fn[lab] else 0.0
        F = 2 * P * R / (P + R) if P + R else 0.0
        per[lab] = {"P": P, "R": R, "F1": F, "tp": tp[lab], "fp": fp[lab], "fn": fn[lab]}
    k = max(len(labels), 1)
    macro = {m: sum(v[m] for v in per.values()) / k for m in ("P", "R", "F1")}
    return macro, per


# ----------------------------------------------------------------------------
# Validation and packaging
# ----------------------------------------------------------------------------

def validate(pred_raw, texts, labels=LABELS):
    """Return a list of problems with a prediction dict (empty list = OK)."""
    errs = []
    if not isinstance(pred_raw, dict):
        return ["top level must be an object {doc_id: [[start, end, label], ...]}"]
    ids, want = set(pred_raw), set(texts)
    if want - ids:
        errs.append(f"missing {len(want - ids)} doc ids, e.g. {sorted(want - ids)[:5]}")
    if ids - want:
        errs.append(f"{len(ids - want)} unknown doc ids, e.g. {sorted(ids - want)[:5]}")
    lab_set = set(labels)
    for doc, spans in pred_raw.items():
        if not isinstance(spans, list):
            errs.append(f"{doc}: value must be a list"); continue
        n = len(texts.get(doc, ""))
        seen = set()
        for sp in spans:
            if not isinstance(sp, list) or len(sp) not in (3, 4):
                errs.append(f"{doc}: bad entry {sp!r}"); continue
            s, e, lab = sp[0], sp[1], sp[2]
            if not (isinstance(s, int) and isinstance(e, int)) or isinstance(s, bool):
                errs.append(f"{doc}: offsets must be ints {sp!r}"); continue
            if not 0 <= s < e <= n:
                errs.append(f"{doc}: offsets out of range {sp!r} (len {n})")
            if lab not in lab_set:
                errs.append(f"{doc}: unknown label {lab!r}")
            if len(sp) == 4 and not isinstance(sp[3], str):
                errs.append(f"{doc}: optional 4th field must be the entity text {sp!r}")
            if (s, e, lab) in seen:
                errs.append(f"{doc}: duplicate span {sp!r}")
            seen.add((s, e, lab))
    return errs


def make_zip(pred_path, out_path="submission.zip"):
    """Write prediction.json directly at the root of the archive."""
    with zipfile.ZipFile(out_path, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(pred_path, arcname="prediction.json")
    return out_path


def write_predictions(pred, path="prediction.json", texts=None):
    """pred: {doc_id: [(s, e, label)]}. Writes [[s, e, label]] lists (sorted)."""
    out = {d: [[int(s), int(e), l] for s, e, l in sorted(set(v))] for d, v in pred.items()}
    if texts is not None:
        for d in texts:
            out.setdefault(d, [])
        errs = validate(out, texts)
        if errs:
            raise ValueError("invalid predictions:\n" + "\n".join(errs[:20]))
    with open(path, "w", encoding="utf-8") as f:
        json.dump(out, f, ensure_ascii=False)
    return path


# ----------------------------------------------------------------------------
# CLI
# ----------------------------------------------------------------------------

def _print_scores(macro, per):
    print(f"{'label':<10} {'P':>6} {'R':>6} {'F1':>6} {'tp':>6} {'fp':>6} {'fn':>6}")
    for lab, v in per.items():
        print(f"{lab:<10} {v['P']:6.3f} {v['R']:6.3f} {v['F1']:6.3f} {v['tp']:6d} {v['fp']:6d} {v['fn']:6d}")
    print(f"{'MACRO':<10} {macro['P']:6.3f} {macro['R']:6.3f} {macro['F1']:6.3f}")


def main(argv):
    if len(argv) < 2:
        print(__doc__); return 1
    cmd = argv[1]
    if cmd == "roundtrip":
        texts, gold = load_texts(argv[2]), load_spans(argv[3])
        pred, bad, total = {}, 0, 0
        for d, spans in gold.items():
            offs = word_offsets(texts[d])
            tags, mis = spans_to_bio(offs, spans)
            pred[d] = bio_to_spans(offs, tags)
            bad += len(mis); total += len(spans)
        print(f"spans not on word boundaries: {bad}/{total}")
        _print_scores(*strict_scores(gold, pred))
    elif cmd == "score":
        _print_scores(*strict_scores(load_spans(argv[2]), load_spans(argv[3])))
    elif cmd == "validate":
        texts = load_texts(argv[2])
        with open(argv[3], encoding="utf-8") as f:
            errs = validate(json.load(f), texts)
        print("OK" if not errs else "\n".join(errs[:50]))
        return 1 if errs else 0
    elif cmd == "zip":
        print(make_zip(argv[2], argv[3] if len(argv) > 3 else "submission.zip"))
    else:
        print(__doc__); return 1
    return 0


if __name__ == "__main__":
    sys.exit(main(sys.argv))

In [ ]:
%%writefile lner_model.py
"""Phase 2 (L-NER) neural pipeline: tokenizer -> encoder -> BIO tagger.

Model-agnostic on purpose. Anything that maps input_ids [B, T] (+ mask) to
hidden states [B, T, H] can be dropped in as the encoder, so the same loop
trains the from-scratch baselines here and, later, the Phase 1 foundation
model(s) (see `load_phase1_encoder` and `CallableEncoder`).

Tokenizer interface: `encode(text) -> (ids, offsets)` with character offsets
into the original string. `BPETokenizer` wraps a Hugging Face `tokenizers`
byte-level BPE saved as tokenizer.json, which is the format the Phase 1 thread
settled on (vocab 32k, pad=0 unk=1 <s>=2 </s>=3 mask=4 sep=5 + extra_id_*,
ByteLevel pre-tokenizer, add_prefix_space=False, trim_offsets=True).

Everything is trained from random initialisation: no pretrained weights,
no external data, no LLM APIs.

CLI (run with python3 -I when DATA_DIR holds downloaded files):
  # train a stand-in tokenizer on the training folds (same settings as Phase 1)
  python lner_model.py tokenizer DATA_DIR OUT/tokenizer.json [--vocab 32000]
  # train on fold1, pick the best epoch on fold2 (strict macro F1)
  python lner_model.py train DATA_DIR RUN_DIR --tokenizer TOK.json [--encoder transformer|bilstm] [--full]
  # predict fold3 -> RUN_DIR/prediction.json + submission.zip (validated)
  python lner_model.py predict DATA_DIR RUN_DIR
"""

import argparse
import json
import math
import os
import random
import sys
import time

import torch
import torch.nn as nn
import torch.nn.functional as F

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import lner_utils as U  # noqa: E402

SEED = 42
TAGS = ["O"] + [f"{p}-{l}" for l in U.LABELS for p in ("B", "I")]
TAG2ID = {t: i for i, t in enumerate(TAGS)}
IGNORE = -100

SPECIAL_TOKENS = ["<pad>", "<unk>", "<s>", "</s>", "<mask>", "<sep>"] + [f"<extra_id_{i}>" for i in range(64)]


def set_seed(seed=SEED):
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


# ----------------------------------------------------------------------------
# Tokenizer
# ----------------------------------------------------------------------------

def train_bpe(texts, out_path, vocab_size=32000):
    """Byte-level BPE with the Phase 1 settings. Stand-in until the Phase 1
    tokenizer.json exists; only the training text differs."""
    from tokenizers import Tokenizer, decoders, models, pre_tokenizers, processors, trainers
    tok = Tokenizer(models.BPE(unk_token="<unk>"))
    tok.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    tok.post_processor = processors.ByteLevel(trim_offsets=True)
    tok.decoder = decoders.ByteLevel()
    trainer = trainers.BpeTrainer(vocab_size=vocab_size, special_tokens=SPECIAL_TOKENS,
                                  initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),
                                  show_progress=False)
    tok.train_from_iterator(list(texts), trainer=trainer)
    os.makedirs(os.path.dirname(os.path.abspath(out_path)), exist_ok=True)
    tok.save(out_path)
    return out_path


class BPETokenizer:
    """encode(text) -> (ids, [(start, end)]) for a tokenizers tokenizer.json."""

    def __init__(self, path, split_punct=True):
        from tokenizers import Tokenizer
        self.path = path
        self.tok = Tokenizer.from_file(path)
        self.vocab_size = self.tok.get_vocab_size()
        self.pad_id = self.tok.token_to_id("<pad>") or 0
        self.split_punct = split_punct

    def encode(self, text):
        if not self.split_punct:
            enc = self.tok.encode(text, add_special_tokens=False)
            return enc.ids, enc.offsets
        # The GPT-2 style ByteLevel regex keeps runs of punctuation together
        # (".," or ".)"), so a gold span like "Cr.P.C." followed by "," would
        # end inside a token. Encode each word / punctuation mark with its
        # leading whitespace instead: words get the same ids as in running
        # text, and every punctuation mark becomes its own token boundary.
        pieces, starts, prev = [], [], 0
        for s, e in U.word_offsets(text):
            pieces.append(text[prev:e]); starts.append(prev); prev = e
        if not pieces:
            return [], []
        ids, offs = [], []
        for enc, base in zip(self.tok.encode_batch(pieces, add_special_tokens=False), starts):
            ids += enc.ids
            offs += [(base + a, base + b) for a, b in enc.offsets]
        # trim_offsets does not trim the leading space of a sequence's first
        # token, which is every piece here, so trim whitespace ourselves.
        out = []
        for a, b in offs:
            while a < b and text[a].isspace(): a += 1
            while b > a and text[b - 1].isspace(): b -= 1
            out.append((a, b))
        return ids, out


# ----------------------------------------------------------------------------
# Span <-> token tags (zero-length tokens, e.g. trimmed whitespace, are ignored)
# ----------------------------------------------------------------------------

def token_tags(offsets, spans):
    """Tag ids per token; zero-length tokens get IGNORE. Returns (ids, n_misaligned)."""
    keep = [i for i, (s, e) in enumerate(offsets) if e > s]
    tags, mis = U.spans_to_bio([offsets[i] for i in keep], spans)
    out = [IGNORE] * len(offsets)
    for i, t in zip(keep, tags):
        out[i] = TAG2ID[t]
    return out, len(mis)


def decode_tags(offsets, tag_ids):
    keep = [i for i, (s, e) in enumerate(offsets) if e > s]
    return U.bio_to_spans([offsets[i] for i in keep], [TAGS[tag_ids[i]] for i in keep])


def _allowed_transitions():
    """allowed[i, j]: tag j may follow tag i under BIO (I-X only after B-X/I-X)."""
    n = len(TAGS)
    allowed = torch.ones(n, n, dtype=torch.bool)
    start = torch.ones(n, dtype=torch.bool)
    for j, t in enumerate(TAGS):
        if t.startswith("I-"):
            lab = t[2:]
            start[j] = False
            for i, u in enumerate(TAGS):
                allowed[i, j] = u in (f"B-{lab}", f"I-{lab}")
    return allowed, start


_ALLOWED, _START = _allowed_transitions()


def viterbi_bio(logp):
    """Best BIO-valid tag sequence for log-probs [T, n_tags] (no learned CRF)."""
    T, n = logp.shape
    if T == 0:
        return []
    neg = torch.finfo(logp.dtype).min / 2
    trans = torch.where(_ALLOWED, torch.zeros(()), torch.full((), neg))
    score = torch.where(_START, logp[0], torch.full((), neg))
    back = torch.zeros(T, n, dtype=torch.long)
    for t in range(1, T):
        cand = score[:, None] + trans  # [from, to]
        score, back[t] = cand.max(0)
        score = score + logp[t]
    best = [int(score.argmax())]
    for t in range(T - 1, 0, -1):
        best.append(int(back[t, best[-1]]))
    return best[::-1]


# ----------------------------------------------------------------------------
# Encoders (all from random init). Any module with .hidden_size and
# forward(input_ids, attention_mask) -> [B, T, H] works.
# ----------------------------------------------------------------------------

class TransformerEncoder(nn.Module):
    """Small pre-norm bidirectional transformer with learned positions."""

    def __init__(self, vocab_size, hidden=256, layers=4, heads=4, ff=1024, max_len=1024, dropout=0.1, pad_id=0):
        super().__init__()
        self.hidden_size = hidden
        self.tok = nn.Embedding(vocab_size, hidden, padding_idx=pad_id)
        self.pos = nn.Embedding(max_len, hidden)
        layer = nn.TransformerEncoderLayer(hidden, heads, ff, dropout, activation="gelu",
                                           batch_first=True, norm_first=True)
        self.layers = nn.TransformerEncoder(layer, layers, enable_nested_tensor=False)
        self.norm = nn.LayerNorm(hidden)
        self.drop = nn.Dropout(dropout)

    def forward(self, input_ids, attention_mask):
        pos = torch.arange(input_ids.size(1), device=input_ids.device)
        x = self.drop(self.tok(input_ids) + self.pos(pos)[None])
        x = self.layers(x, src_key_padding_mask=~attention_mask.bool())
        return self.norm(x)


class BiLSTMEncoder(nn.Module):
    """Embedding + BiLSTM; trains well on little data and runs fine on CPU."""

    def __init__(self, vocab_size, hidden=256, layers=2, emb=256, dropout=0.3, pad_id=0, **_):
        super().__init__()
        self.hidden_size = hidden
        self.tok = nn.Embedding(vocab_size, emb, padding_idx=pad_id)
        self.drop = nn.Dropout(dropout)
        self.lstm = nn.LSTM(emb, hidden // 2, layers, batch_first=True, bidirectional=True,
                            dropout=dropout if layers > 1 else 0.0)

    def forward(self, input_ids, attention_mask):
        x = self.drop(self.tok(input_ids))
        lengths = attention_mask.sum(1).clamp(min=1).cpu()
        packed = nn.utils.rnn.pack_padded_sequence(x, lengths, batch_first=True, enforce_sorted=False)
        out, _ = self.lstm(packed)
        out, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True, total_length=input_ids.size(1))
        return out


class CallableEncoder(nn.Module):
    """Wrap a Phase 1 model as an encoder.

    fn(module, input_ids, attention_mask) -> [B, T, H]. For a Hugging Face
    style model: lambda m, ids, am: m(input_ids=ids, attention_mask=am,
    output_hidden_states=True).hidden_states[-1]. For a decoder-only (causal)
    model each token only sees its left context; consider running it on the
    text and its reverse and concatenating (hidden_size doubles).
    """

    def __init__(self, module, hidden_size, fn):
        super().__init__()
        self.module, self.hidden_size, self.fn = module, hidden_size, fn

    def forward(self, input_ids, attention_mask):
        return self.fn(self.module, input_ids, attention_mask)


def load_phase1_encoder(module, ckpt_path, strip_prefix="", map_location="cpu"):
    """Load Phase 1 weights into `module` (strict=False) and report what did not match."""
    sd = torch.load(ckpt_path, map_location=map_location)
    if isinstance(sd, dict) and "state_dict" in sd:
        sd = sd["state_dict"]
    if isinstance(sd, dict) and "model" in sd and isinstance(sd["model"], dict):
        sd = sd["model"]
    if strip_prefix:
        sd = {k[len(strip_prefix):] if k.startswith(strip_prefix) else k: v for k, v in sd.items()}
    res = module.load_state_dict(sd, strict=False)
    print(f"loaded {ckpt_path}: {len(res.missing_keys)} missing, {len(res.unexpected_keys)} unexpected keys")
    return res


def phase1_encoder(ckpt_path, p1_dir=None, device="cpu"):
    """Encoder from a Phase 1 export (model_final.pt / model_best.pt / last.pt),
    using the model code in the Phase 1 module cs781_p1.py (`load_model`).

    Feature extractor per architecture (see cs781_p1 docstrings):
      gpt    DecoderLM.hidden, causal (left context only)
      unilm  DecoderLM.hidden with mode=1, causal=False (bidirectional)
      encdec EncDecLM.encode (bidirectional encoder only; decoder unused)
      retnet RetNetLM.hidden (causal decay retention)
    Our batches are right-padded; padding is masked for the bidirectional
    archs and cannot leak into real tokens for the causal ones.
    """
    if p1_dir:
        sys.path.insert(0, p1_dir)
    import cs781_p1 as P1
    model, mcfg = P1.load_model(ckpt_path, device)
    model.train()
    arch = mcfg.arch
    if arch == "gpt":
        fn = lambda m, ids, am: m.hidden(ids, causal=True, mode=0)
    elif arch == "unilm":
        fn = lambda m, ids, am: m.hidden(ids, mask=am.bool()[:, None, None, :], causal=False, mode=1)
    elif arch == "encdec":
        fn = lambda m, ids, am: m.encode(ids, am.bool())
    elif arch == "retnet":
        fn = lambda m, ids, am: m.hidden(ids, am.bool())[0]
    else:
        raise ValueError(f"unknown Phase 1 arch {arch!r}")
    enc = CallableEncoder(model, mcfg.d_model, fn)
    enc.arch = arch
    return enc


ENCODERS = {"transformer": TransformerEncoder, "bilstm": BiLSTMEncoder}


class Tagger(nn.Module):
    def __init__(self, encoder, n_tags=len(TAGS), dropout=0.1):
        super().__init__()
        self.encoder = encoder
        self.drop = nn.Dropout(dropout)
        self.head = nn.Linear(encoder.hidden_size, n_tags)

    def forward(self, input_ids, attention_mask):
        return self.head(self.drop(self.encoder(input_ids, attention_mask)))


# ----------------------------------------------------------------------------
# Data
# ----------------------------------------------------------------------------

def encode_docs(tokenizer, texts, gold=None):
    """{doc: (ids, offsets, tag_ids|None)} and alignment stats."""
    out, mis, total = {}, 0, 0
    for d, t in texts.items():
        ids, offs = tokenizer.encode(t)
        tags = None
        if gold is not None:
            spans = gold.get(d, [])
            tags, m = token_tags(offs, spans)
            mis += m; total += len(spans)
        out[d] = (ids, offs, tags)
    return out, (mis, total)


def make_windows(encoded, max_len, stride):
    ex = []
    for d, (ids, _, tags) in encoded.items():
        for lo, hi in U.windows(len(ids), max_len, stride):
            ex.append((ids[lo:hi], tags[lo:hi]))
    return ex


def collate(batch, pad_id):
    T = max(len(i) for i, _ in batch)
    ids = torch.full((len(batch), T), pad_id, dtype=torch.long)
    lab = torch.full((len(batch), T), IGNORE, dtype=torch.long)
    am = torch.zeros((len(batch), T), dtype=torch.long)
    for b, (i, t) in enumerate(batch):
        ids[b, :len(i)] = torch.tensor(i)
        lab[b, :len(t)] = torch.tensor(t)
        am[b, :len(i)] = 1
    return ids, am, lab


# ----------------------------------------------------------------------------
# Train / predict
# ----------------------------------------------------------------------------

@torch.no_grad()
def predict_docs(model, encoded, max_len, stride, device, batch_size=16):
    """Average log-probs over overlapping windows, then BIO-constrained Viterbi."""
    model.eval()
    jobs = []  # (doc, lo, hi)
    acc = {}
    for d, (ids, _, _) in encoded.items():
        acc[d] = (torch.zeros(len(ids), len(TAGS)), torch.zeros(len(ids), 1))
        jobs += [(d, lo, hi) for lo, hi in U.windows(len(ids), max_len, stride)]
    pad = getattr(model, "pad_id", 0)
    for k in range(0, len(jobs), batch_size):
        chunk = jobs[k:k + batch_size]
        batch = [(encoded[d][0][lo:hi], [IGNORE] * (hi - lo)) for d, lo, hi in chunk]
        ids, am, _ = collate(batch, pad)
        with torch.autocast(device_type=device.type, dtype=torch.bfloat16, enabled=device.type == "cuda"):
            logits = model(ids.to(device), am.to(device))
        logp = F.log_softmax(logits.float(), -1).cpu()
        for b, (d, lo, hi) in enumerate(chunk):
            s, c = acc[d]
            s[lo:hi] += logp[b, :hi - lo]
            c[lo:hi] += 1
    pred = {}
    for d, (ids, offs, _) in encoded.items():
        s, c = acc[d]
        tag_ids = viterbi_bio(s / c.clamp(min=1))
        pred[d] = decode_tags(offs, tag_ids)
    return pred


def build_model(cfg, vocab_size, pad_id):
    enc = ENCODERS[cfg["encoder"]](vocab_size, pad_id=pad_id, **cfg["encoder_args"])
    model = Tagger(enc, dropout=cfg.get("head_dropout", 0.1))
    model.pad_id = pad_id
    return model


def train(cfg, tokenizer, train_texts, train_gold, dev_texts=None, dev_gold=None, log=print, model=None):
    """Returns (model, history). Keeps the weights of the best dev epoch."""
    set_seed(cfg.get("seed", SEED))
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    enc_train, (mis, tot) = encode_docs(tokenizer, train_texts, train_gold)
    log(f"train docs {len(enc_train)}, gold spans {tot}, not on token boundaries {mis}")
    examples = make_windows(enc_train, cfg["max_len"], cfg["stride"])
    enc_dev = encode_docs(tokenizer, dev_texts)[0] if dev_texts else None
    if model is None:
        model = build_model(cfg, tokenizer.vocab_size, tokenizer.pad_id)
    model.to(device)
    log(f"params {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M, windows {len(examples)}, device {device}")
    # Optional lower learning rate for a pretrained (Phase 1) encoder than for the new head.
    enc_params = [p for p in model.encoder.parameters() if p.requires_grad]
    enc_ids = {id(p) for p in enc_params}
    rest = [p for p in model.parameters() if p.requires_grad and id(p) not in enc_ids]
    groups = [{"params": enc_params, "lr": cfg.get("encoder_lr", cfg["lr"])}, {"params": rest, "lr": cfg["lr"]}]
    opt = torch.optim.AdamW(groups, weight_decay=cfg.get("weight_decay", 0.01))
    steps = cfg["epochs"] * math.ceil(len(examples) / cfg["batch_size"])
    warm = int(cfg.get("warmup", 0.06) * steps)
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: min(1.0, (s + 1) / max(1, warm)) * max(0.0, (steps - s) / max(1, steps - warm)) if s >= warm
        else (s + 1) / max(1, warm))
    # Down-weight "O" a little: entities are a small share of tokens.
    weight = torch.ones(len(TAGS)); weight[0] = cfg.get("o_weight", 1.0)
    weight = weight.to(device)
    best, best_state, hist = -1.0, None, []
    for ep in range(1, cfg["epochs"] + 1):
        model.train(); random.shuffle(examples); t0 = time.time(); tot_loss = 0.0
        for k in range(0, len(examples), cfg["batch_size"]):
            ids, am, lab = collate(examples[k:k + cfg["batch_size"]], tokenizer.pad_id)
            ids, am, lab = ids.to(device), am.to(device), lab.to(device)
            with torch.autocast(device_type=device.type, dtype=torch.bfloat16, enabled=device.type == "cuda"):
                logits = model(ids, am)
            loss = F.cross_entropy(logits.float().view(-1, len(TAGS)), lab.view(-1), weight=weight, ignore_index=IGNORE)
            opt.zero_grad(set_to_none=True); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sched.step(); tot_loss += loss.item()
        row = {"epoch": ep, "loss": tot_loss / math.ceil(len(examples) / cfg["batch_size"]), "sec": time.time() - t0}
        if enc_dev is not None:
            macro, _ = U.strict_scores(dev_gold, predict_docs(model, enc_dev, cfg["max_len"], cfg["stride"], device))
            row["dev_f1"] = macro["F1"]
            if macro["F1"] > best:
                best = macro["F1"]
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        hist.append(row); log(json.dumps(row))
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, hist


DEFAULT_CFG = {
    "encoder": "transformer",
    "encoder_args": {"hidden": 256, "layers": 4, "heads": 4, "ff": 1024, "max_len": 1024, "dropout": 0.1},
    "max_len": 512, "stride": 384, "batch_size": 16, "epochs": 20, "lr": 5e-4,
    "weight_decay": 0.01, "warmup": 0.06, "o_weight": 0.5, "seed": SEED,
}
BILSTM_ARGS = {"hidden": 384, "layers": 2, "emb": 256, "dropout": 0.3}


def _folds(data):
    p = lambda n: os.path.join(data, n)
    return (U.load_texts(p("fold1-text.json")), U.load_spans(p("fold1-gold.json")),
            U.load_texts(p("fold2-text.json")), U.load_spans(p("fold2-gold.json")))


def main(argv=None):
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("cmd", choices=["tokenizer", "train", "predict"])
    ap.add_argument("data")
    ap.add_argument("out")
    ap.add_argument("--tokenizer")
    ap.add_argument("--vocab", type=int, default=32000)
    ap.add_argument("--encoder", default="transformer", choices=list(ENCODERS))
    ap.add_argument("--epochs", type=int)
    ap.add_argument("--lr", type=float)
    ap.add_argument("--full", action="store_true", help="train on fold1+fold2 (no dev selection)")
    a = ap.parse_args(argv)
    # Resolve paths up front: a long run on a remounted shared folder can lose its cwd.
    a.data, a.out = os.path.abspath(a.data), os.path.abspath(a.out)
    if a.tokenizer:
        a.tokenizer = os.path.abspath(a.tokenizer)

    if a.cmd == "tokenizer":
        t1, _, t2, _ = _folds(a.data)
        texts = list(t1.values()) + list(t2.values())
        print(train_bpe(texts, a.out, a.vocab))
        return 0

    if a.cmd == "train":
        cfg = json.loads(json.dumps(DEFAULT_CFG))
        cfg["encoder"] = a.encoder
        if a.encoder == "bilstm":
            cfg["encoder_args"] = dict(BILSTM_ARGS); cfg["lr"] = 2e-3
        if a.epochs: cfg["epochs"] = a.epochs
        if a.lr: cfg["lr"] = a.lr
        tok = BPETokenizer(a.tokenizer)
        t1, g1, t2, g2 = _folds(a.data)
        os.makedirs(a.out, exist_ok=True)
        if a.full:
            model, hist = train(cfg, tok, {**t1, **t2}, {**g1, **g2})
        else:
            model, hist = train(cfg, tok, t1, g1, t2, g2)
        cfg["tokenizer"] = a.tokenizer
        torch.save(model.state_dict(), os.path.join(a.out, "model.pt"))
        json.dump({"cfg": cfg, "history": hist}, open(os.path.join(a.out, "run.json"), "w"), indent=1)
        return 0

    if a.cmd == "predict":
        run = json.load(open(os.path.join(a.out, "run.json")))
        cfg = run["cfg"]
        tok = BPETokenizer(a.tokenizer or cfg["tokenizer"])
        model = build_model(cfg, tok.vocab_size, tok.pad_id)
        model.load_state_dict(torch.load(os.path.join(a.out, "model.pt"), map_location="cpu"))
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        model.to(device)
        t3 = U.load_texts(os.path.join(a.data, "fold3-text.json"))
        pred = predict_docs(model, encode_docs(tok, t3)[0], cfg["max_len"], cfg["stride"], device)
        path = U.write_predictions(pred, os.path.join(a.out, "prediction.json"), texts=t3)
        print(U.make_zip(path, os.path.join(a.out, "submission.zip")))
        return 0


if __name__ == "__main__":
    sys.exit(main())

In [ ]:
%%writefile baseline_rules.py
"""Phase 2 baseline without any neural model: a gazetteer learned from the
training gold spans plus a few regexes. Pure Python (needs only lner_utils).

It exists to (1) validate the whole data -> prediction.json -> zip path on the
real files and (2) give a quick first Codabench submission. It is not a
"Phase 1 model" experiment and should be described as a non-neural baseline.

Usage:
  # dev run: learn from fold1, score on fold2 (prints strict macro P/R/F1)
  python baseline_rules.py dev  DATA_DIR
  # submission: learn from fold1+fold2, predict fold3, validate, zip
  python baseline_rules.py test DATA_DIR OUT_DIR

DATA_DIR holds fold1-text.json, fold1-gold.json, fold2-text.json,
fold2-gold.json, fold3-text.json (labels.txt optional).
"""

import os
import re
import sys
from collections import Counter, defaultdict

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import lner_utils as U  # noqa: E402

MIN_COUNT = 1        # gazetteer entry must be seen this many times
MIN_PURITY = 0.6     # share of the entry's majority label among its gold uses
MIN_LEN = 3          # ignore very short surface strings ("Sh", "Ld")

_MONTHS = (r"(?:January|February|March|April|May|June|July|August|September|"
           r"October|November|December|Jan|Feb|Mar|Apr|Jun|Jul|Aug|Sep|Sept|Oct|Nov|Dec)\.?")
REGEXES = [
    ("DATE", re.compile(rf"\b\d{{1,2}}(?:st|nd|rd|th)?\.?\s+{_MONTHS},?\s+\d{{4}}\b")),
    ("DATE", re.compile(rf"\b{_MONTHS}\s+\d{{1,2}}(?:st|nd|rd|th)?,?\s+\d{{4}}\b")),
    ("DATE", re.compile(r"\b\d{1,2}[./-]\d{1,2}[./-](?:19|20)\d{2}\b")),
    ("CASENO", re.compile(r"\b(?:Criminal|Civil|Crl\.?|Writ|W\.P\.|S\.L\.P\.|SLP)[\w.() ]{0,40}?"
                          r"No\.?\s*\d+(?:\s*(?:/|of)\s*\d{2,4})?", re.I)),
    ("STAT", re.compile(r"\b(?:Indian Penal Code|Code of Criminal Procedure|Constitution of India|"
                        r"Code of Civil Procedure|Evidence Act|[A-Z][\w.,' ]{2,80}? Act,? (?:18|19|20)\d{2})\b")),
]


def build_gazetteer(texts, gold):
    """surface string -> majority label, kept only if frequent and unambiguous."""
    by_surface = defaultdict(Counter)
    for d, spans in gold.items():
        t = texts[d]
        for s, e, lab in spans:
            by_surface[t[s:e]][lab] += 1
    gaz = {}
    for surf, c in by_surface.items():
        lab, n = c.most_common(1)[0]
        if len(surf.strip()) >= MIN_LEN and sum(c.values()) >= MIN_COUNT and n / sum(c.values()) >= MIN_PURITY:
            gaz[surf] = lab
    return gaz


def _gazetteer_regex(gaz):
    # Longest first so "Supreme Court of India" wins over "Supreme Court".
    alts = sorted(gaz, key=len, reverse=True)
    if not alts:
        return None
    return re.compile(r"(?<!\w)(?:" + "|".join(re.escape(a) for a in alts) + r")(?!\w)")


def predict_doc(text, gaz, gaz_re):
    cands = []  # (start, end, label, priority)
    if gaz_re is not None:
        for m in gaz_re.finditer(text):
            cands.append((m.start(), m.end(), gaz[m.group(0)], 0))
    for lab, rx in REGEXES:
        for m in rx.finditer(text):
            s, e = m.start(), m.end()
            while e > s and text[e - 1] in " ,;:": e -= 1
            cands.append((s, e, lab, 1))
    # Resolve overlaps: gazetteer before regex, then longer spans first.
    cands.sort(key=lambda x: (x[3], -(x[1] - x[0]), x[0]))
    taken, out = [], []
    for s, e, lab, _ in cands:
        if any(s < te and ts < e for ts, te in taken):
            continue
        taken.append((s, e)); out.append((s, e, lab))
    return sorted(out)


def predict(texts, gaz):
    gaz_re = _gazetteer_regex(gaz)
    return {d: predict_doc(t, gaz, gaz_re) for d, t in texts.items()}


def main(argv):
    if len(argv) < 3 or argv[1] not in ("dev", "test"):
        print(__doc__); return 1
    data = argv[2]
    p = lambda name: os.path.join(data, name)
    t1, g1 = U.load_texts(p("fold1-text.json")), U.load_spans(p("fold1-gold.json"))
    t2, g2 = U.load_texts(p("fold2-text.json")), U.load_spans(p("fold2-gold.json"))
    if argv[1] == "dev":
        pred = predict(t2, build_gazetteer(t1, g1))
        U._print_scores(*U.strict_scores(g2, pred))
        return 0
    out = argv[3] if len(argv) > 3 else "."
    os.makedirs(out, exist_ok=True)
    t3 = U.load_texts(p("fold3-text.json"))
    pred = predict(t3, build_gazetteer({**t1, **t2}, {**g1, **g2}))
    path = U.write_predictions(pred, os.path.join(out, "prediction.json"), texts=t3)
    print(U.make_zip(path, os.path.join(out, "submission.zip")))
    return 0


if __name__ == "__main__":
    sys.exit(main(sys.argv))

## 2. Data

In [ ]:
import os, json, sys
from kaggle_secrets import UserSecretsClient
from huggingface_hub import snapshot_download
HF_TOKEN = UserSecretsClient().get_secret("HF")
root = snapshot_download("Exploration-Lab/CS-781-Capstone", repo_type="dataset",
                         allow_patterns=["Phase_2_LNER/*"], token=HF_TOKEN, local_dir=f"{WORK}/hf")
DATA = f"{WORK}/p2data"
os.makedirs(DATA, exist_ok=True)
for sub in ("Train", "Test"):
    d = os.path.join(root, "Phase_2_LNER", sub)
    for f in os.listdir(d):
        os.replace(os.path.join(d, f), os.path.join(DATA, f))
print(sorted(os.listdir(DATA)))

In [ ]:
import importlib, lner_utils as U, lner_model as M, baseline_rules as R
from collections import Counter
t1, g1 = U.load_texts(f"{DATA}/fold1-text.json"), U.load_spans(f"{DATA}/fold1-gold.json")
t2, g2 = U.load_texts(f"{DATA}/fold2-text.json"), U.load_spans(f"{DATA}/fold2-gold.json")
t3 = U.load_texts(f"{DATA}/fold3-text.json")
print("labels.txt:", U.load_labels(f"{DATA}/labels.txt"))
for name, t, g in [("fold1", t1, g1), ("fold2", t2, g2), ("fold3", t3, None)]:
    chars = sum(map(len, t.values()))
    print(name, "docs", len(t), "chars", chars, "spans", sum(map(len, g.values())) if g else "-")
print("label counts fold1+2:", Counter(l for g in (g1, g2) for v in g.values() for *_, l in v).most_common())

## 3. Tokenizer
The Phase 1 tokenizer if available, otherwise a stand-in byte-level BPE with the same settings trained on fold1+fold2 text.

In [ ]:
TOK = PHASE1_TOKENIZER or M.train_bpe(list(t1.values()) + list(t2.values()), f"{WORK}/tokenizer_standin.json", 32000)
tok = M.BPETokenizer(TOK)
# Round trip: gold -> token BIO -> spans. This is the ceiling for any tagger on this tokenization.
for name, t, g in [("fold1", t1, g1), ("fold2", t2, g2)]:
    enc, (mis, tot) = M.encode_docs(tok, t, g)
    back = {d: M.decode_tags(o, [x if x != M.IGNORE else 0 for x in tg]) for d, (_, o, tg) in enc.items()}
    print(name, f"spans not on token boundaries {mis}/{tot}", "round-trip macro F1 %.4f" % U.strict_scores(g, back)[0]["F1"])

## 4. Non-neural baseline (gazetteer + regex)
A quick, always-valid submission and a reference point. Not a Phase 1 experiment.

In [ ]:
pred = R.predict(t2, R.build_gazetteer(t1, g1))
U._print_scores(*U.strict_scores(g2, pred))
pred3 = R.predict(t3, R.build_gazetteer({**t1, **t2}, {**g1, **g2}))
os.makedirs(f"{WORK}/rules", exist_ok=True)
U.make_zip(U.write_predictions(pred3, f"{WORK}/rules/prediction.json", texts=t3), f"{WORK}/submission_rules.zip")

## 5. Neural tagger: train on fold1, select epoch on fold2

In [ ]:
import copy, torch
cfg = copy.deepcopy(M.DEFAULT_CFG)
cfg["encoder"] = ENCODER
if ENCODER == "bilstm":
    cfg["encoder_args"] = dict(M.BILSTM_ARGS); cfg["lr"] = 2e-3
cfg["epochs"] = DEV_EPOCHS
model, hist = M.train(cfg, tok, t1, g1, t2, g2)
best_ep = max(hist, key=lambda r: r["dev_f1"])["epoch"]
print("best epoch", best_ep, "dev F1", max(r["dev_f1"] for r in hist))
dev_pred = M.predict_docs(model, M.encode_docs(tok, t2)[0], cfg["max_len"], cfg["stride"], next(model.parameters()).device)
U._print_scores(*U.strict_scores(g2, dev_pred))

## 6. Final model on fold1+fold2, predict fold3, validate, zip

In [ ]:
cfg_full = dict(cfg, epochs=best_ep)
model_full, _ = M.train(cfg_full, tok, {**t1, **t2}, {**g1, **g2})
dev = next(model_full.parameters()).device
pred3 = M.predict_docs(model_full, M.encode_docs(tok, t3)[0], cfg["max_len"], cfg["stride"], dev)
out = f"{WORK}/{RUN_NAME}"; os.makedirs(out, exist_ok=True)
torch.save(model_full.state_dict(), f"{out}/model.pt")
json.dump({"cfg": dict(cfg_full, tokenizer=TOK), "dev_history": hist}, open(f"{out}/run.json", "w"), indent=1)
path = U.write_predictions(pred3, f"{out}/prediction.json", texts=t3)   # raises if invalid
print(U.make_zip(path, f"{WORK}/submission_{RUN_NAME}.zip"))

## 7. Hook: fine-tune a Phase 1 model
`M.phase1_encoder` loads any of the four Phase 1 exports with `cs781_p1.load_model` and picks the feature extractor per architecture: `unilm` bidirectional mode, the `encdec` encoder, causal `gpt` and `retnet` hidden states. Set `PHASE1_TOKENIZER` to the Phase 1 tokenizer.json too. Full fine-tuning here; LORMA and the other PEFT and distillation experiments build on the same `M.train(..., model=...)` call.

In [ ]:
if PHASE1_CKPT:
    # cs781_p1.py comes from the Phase 1 notebook output (attach it as a Kaggle input).
    enc = M.phase1_encoder(PHASE1_CKPT, p1_dir=PHASE1_CODE_DIR)
    print("Phase 1 arch:", enc.arch)   # NER should prefer unilm or encdec (bidirectional)
    p1_model = M.Tagger(enc); p1_model.pad_id = tok.pad_id
    cfg_p1 = dict(cfg, encoder_lr=5e-5, lr=1e-3, epochs=8, batch_size=8)
    p1_model, hist_p1 = M.train(cfg_p1, tok, t1, g1, t2, g2, model=p1_model)
    print("best dev F1", max(r["dev_f1"] for r in hist_p1))

## LLM usage declaration
LLMs were used for code generation and analysis, as permitted by the updated course policy. No external LLM API is called anywhere in this pipeline.

_Fill in: tools used, for which parts, and how outputs were checked._